# CopyWrite AI - run on a free GPU

1. **Runtime → Change runtime type → T4 GPU** (Colab) or **Accelerator → GPU T4** (Kaggle, also turn *Internet* on).
2. Set your GitHub repo URL below and run the cells top to bottom.
3. The last cell prints a public `https://….gradio.live` link - open it on your laptop.

First run downloads the Emuru model (~3 GB, 1-2 min). A free Colab session lasts a few hours; just rerun if it disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/<your-username>/copywrite-ai.git"  # <- change this
!git clone -q $REPO_URL copywrite_repo
%cd copywrite_repo

In [ ]:
!pip install -q -r requirements.txt -r requirements-model.txt
# optional: watermark + metrics
# !pip install -q -r requirements-extra.txt

### Sanity check: does the model load and write a line?

In [ ]:
from huggingface_hub import hf_hub_download
from PIL import Image
from copywrite import EmuruGenerator, StyleRef

gen = EmuruGenerator(); gen.load()
sample = Image.open(hf_hub_download('blowing-up-groundhogs/emuru', 'sample.png')).convert('L')
sample = sample.resize((sample.width * 64 // sample.height, 64))
ref = StyleRef(sample, 'THE JOLLY IS "U"')

import time; t = time.time()
line = gen.generate('Generative AI mini project', ref)
print(f'{time.time()-t:.1f}s'); line

In [ ]:
# OCR check: should print something close to 'THE JOLLY IS "U"'
from copywrite.ocr import transcribe_lines
r = transcribe_lines([sample])[0]
print(r.text, f'{r.confidence:.0%}')

### Launch the web app
Upload your handwriting photo + transcription in the UI. Stop the cell to shut it down.

In [ ]:
!python app.py --share

### (Alternative) command line / evaluation
Upload your files to the `samples/` folder first (Files panel on the left).

In [ ]:
# !python -m copywrite.cli --text samples/text/sample_notes.txt --style samples/style/me.jpg --style-text samples/style/me.txt --backend emuru --out outputs/run1
# !pip install -q -r requirements-extra.txt
# !python scripts/evaluate.py --photos samples/style/me_12lines.jpg --text samples/style/me_12lines.txt --k 2 --backends emuru font --out outputs/eval_me